# Pipeline DINO · comprobación técnica real
Objetivo: verificar construcción, entrenamiento, recuperación de checkpoints y publicación de las tres intervenciones. Pregunta/hipótesis: los componentes preservan C1 inicialmente y pueden entrenarse con gradientes finitos. No es una prueba de mejora estadística. Muestra: ocho fotos train (dos identidades por especie, dos fotos por identidad) del piloto verificado; validación completa de 273 fotos. Semilla 45, una época de dos pasos P2×K2; no test. No elegir hiperparámetros por estos resultados.
Métodos: [DINOv2](https://arxiv.org/abs/2304.07193), [LoRA](https://arxiv.org/abs/2106.09685), [triplet batch-hard](https://arxiv.org/abs/1703.07737). La separación de componentes y selección con época cero son decisiones del estudio. Resultados técnicos; sin inferencia sobre generalización, sesiones o superioridad.
## Fase 1. Preparación y verificación de datos

In [1]:
import sys, os, json, gc, time
from pathlib import Path
if not (3,11) <= sys.version_info[:2] < (3,14):
    raise RuntimeError('Python 3.11–3.13 required')
os.environ['PYTORCH_ENABLE_MPS_FALLBACK']='1'
from dotenv import load_dotenv
load_dotenv('.env')
import numpy as np
import torch
import mlflow
from dataclasses import asdict
from pettrace.data.verification import sha256
from pettrace.tracking import PRODUCTION_URI, source_identity
from pettrace.training.dinov2 import build, Config, Preprocessor, Pipeline, fit
from pettrace.training.lora import load_adapter
from pettrace.evaluation.core import load_model
assert torch.backends.mps.is_available()
mlflow.set_tracking_uri(PRODUCTION_URI);mlflow.set_registry_uri(PRODUCTION_URI)
LOG_TO_MLFLOW=True
REPORT=Path('reports/dino-pipeline-smoke/20261118-mps')
REPORT.mkdir(parents=True,exist_ok=True)
client=mlflow.MlflowClient()
tracking=REPORT/'tracking.json'
if tracking.exists():run_id=json.loads(tracking.read_text())['run_id']
else:
    run_id=client.create_run('3',tags={'mlflow.runName':'DINO pipeline technical check','purpose':'technical_pilot','experiment':'DINO-pipeline'}).info.run_id
    tracking.write_text(json.dumps({'run_id':run_id}))
prior=Path('reports/d1-dino-pilot/20261118-local-v1')
assert sha256(prior/'rows.json')==json.loads((prior/'initial-design.json').read_text())['rows_sha256']
all_rows=json.loads((prior/'rows.json').read_text())
rows=[r for r in all_rows if r['split']=='val']
for sp in ('cat','dog'):
    train=[r for r in all_rows if r['split']=='train' and r['species']==sp]
    for animal in sorted({r['animal'] for r in train})[:2]:
        selected=sorted([r for r in train if r['animal']==animal],key=lambda r:r['sha256'])[:2]
        assert len(selected)==2
        rows+=selected
assert len(rows)==281
for r in rows:assert sha256(r['path'])==r['sha256']
(REPORT/'rows.json').write_text(json.dumps(rows,indent=2))
base=load_model('models:/pettrace-embedder/2');assert base.golden_ok
SOURCE=source_identity()
print('Technical sample verified: 8 train, 273 val; no test',run_id)


2026/11/18 11:13:11 INFO mlflow.agent.hint: Load the `instrumenting-with-mlflow-tracing` skill at /Users/alessandro/Develop/personal/pettrace-ml/.venv/lib/python3.11/site-packages/mlflow/assistant/skills/instrumenting-with-mlflow-tracing/SKILL.md before writing any tracing code; it ships with this MLflow install. Set MLFLOW_DISABLE_AGENT_HINT=1 to silence this.


Technical sample verified: 8 train, 273 val; no test 3fcd5e4922a749768d41fdb283d9b0f5


## Fase 2. Representación y entrenamiento
Comparar proyección, LoRA y combinación conjunta. Sin aumentos. La combinación entrena ambos componentes desde C1; no reproduce la cadena histórica E3→E4 de CLIP. Época cero elegible. Publicación recuperable, con guardado local previo.

In [2]:
results={}
for mode in ('projection','lora','combined'):
    config=Config(seed=45,epochs=1,steps_per_epoch=2,identities_per_batch=2,images_per_identity=2,learning_rate=1e-4 if mode=='projection' else 1e-5)
    folder=REPORT/mode;folder.mkdir(exist_ok=True)
    encoder,processor=build(mode,'mps',config)
    pre=Preprocessor(processor,base.pipeline.detector)
    probe=pre.pixels(rows[-4:])
    with torch.no_grad():
        raw=torch.nn.functional.normalize(encoder.backbone(pixel_values=torch.tensor(probe,device='mps')).last_hidden_state[:,0],dim=-1).cpu().numpy()
        initial=encoder(torch.tensor(probe,device='mps')).cpu().numpy()
    assert np.allclose(initial,raw,atol=1e-6)
    started=time.monotonic()
    if (folder/'training-result.json').exists():
        result=json.loads((folder/'training-result.json').read_text())
        load_adapter(encoder,folder/'adapter.pt')
    else:
        encoder,result=fit(encoder,rows,pre,folder,config,device='mps')
        result['seconds']=time.monotonic()-started
        (folder/'training-result.json').write_text(json.dumps(result,indent=2))
    with torch.no_grad():expected=encoder(torch.tensor(probe,device='mps')).cpu().numpy()
    load_adapter(encoder,folder/'adapter.pt')
    with torch.no_grad():actual=encoder(torch.tensor(probe,device='mps')).cpu().numpy()
    assert np.allclose(actual,expected,atol=1e-6)
    (folder/'design.json').write_text(json.dumps(dict(config=asdict(config),source=SOURCE,mode=mode,rows_sha256=sha256(REPORT/'rows.json'),detector=base.sha256,test_evaluated=False,initial_parity=True,checkpoint_parity=True),indent=2))
    client.log_artifacts(run_id,str(folder),'report/'+mode)
    results[mode]=result
    print(mode,'trainables',result['trainable_parameters'],'initial mAP',np.mean([result['initial_validation'][s]['mAP'] for s in ('cat','dog')]),'selected mAP',result['best_val_mAP'],'epoch',result['best_epoch'],flush=True)
    del encoder,pre,processor
    gc.collect();torch.mps.empty_cache()


/Users/alessandro/Develop/personal/pettrace-ml/.venv/lib/python3.11/site-packages/torch/nn/functional.py:4594: UserWarning: The operator 'aten::upsample_bicubic2d.out' is not currently supported on the MPS backend and will fall back to run on the CPU. This may have performance implications. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/aten/src/ATen/mps/MPSFallback.mm:13.)
  return torch._C._nn.upsample_bicubic2d(


Training seed 45:   0%|          | 0/1 [00:00<?, ?epoch/s]

projection trainables 1048576 initial mAP 0.8387034377752959 selected mAP 0.8387034377752959 epoch 0


Training seed 45:   0%|          | 0/1 [00:00<?, ?epoch/s]

lora trainables 786432 initial mAP 0.8387034377752959 selected mAP 0.838788404054055 epoch 1


Training seed 45:   0%|          | 0/1 [00:00<?, ?epoch/s]

combined trainables 1835008 initial mAP 0.8387034377752959 selected mAP 0.8387845943766823 epoch 1


## Fase 3. Evaluación y análisis
Verificar publicación y guardar resultados técnicos de las tres variantes. Recuperación cerrada de misma especie, leave-one-out por foto. Rank-k por identidad y mAP por fotos. Esta comprobación no evalúa el test público ni PetTrace; no se registra ni promueve un modelo de producción.

In [3]:
summary={mode:{k:r[k] for k in ('best_epoch','best_val_mAP','trainable_parameters','seconds')} for mode,r in results.items()}
(REPORT/'summary.json').write_text(json.dumps(summary,indent=2))
for mode,r in results.items():
    client.log_metric(run_id,mode+'.val.macro_mAP',r['best_val_mAP'])
for f in ('summary.json','rows.json'):client.log_artifact(run_id,str(REPORT/f),'report')
for f in ('dinov2.py','dino_metric.py','lora.py'):client.log_artifact(run_id,'src/pettrace/training/'+f,'source')
client.set_terminated(run_id,'FINISHED')
verified={}
for mode in results:
    path=Path(client.download_artifacts(run_id,'report/'+mode+'/adapter.pt',str(REPORT/'verify'/mode)))
    assert sha256(path)==sha256(REPORT/mode/'adapter.pt')
    verified[mode]=sha256(path)
assert client.get_run(run_id).info.status=='FINISHED'
(REPORT/'publication.json').write_text(json.dumps(dict(published=True,run_id=run_id,verified_checkpoints=verified),indent=2))
client.log_artifact(run_id,str(REPORT/'publication.json'),'report')
print(json.dumps(summary,indent=2));print('Published and verified',run_id)


🏃 View run DINO pipeline technical check at: https://ml.pettrace.app/#/experiments/3/runs/3fcd5e4922a749768d41fdb283d9b0f5
🧪 View experiment at: https://ml.pettrace.app/#/experiments/3


{
  "projection": {
    "best_epoch": 0,
    "best_val_mAP": 0.8387034377752959,
    "trainable_parameters": 1048576,
    "seconds": 28.95981812500395
  },
  "lora": {
    "best_epoch": 1,
    "best_val_mAP": 0.838788404054055,
    "trainable_parameters": 786432,
    "seconds": 28.162252750014886
  },
  "combined": {
    "best_epoch": 1,
    "best_val_mAP": 0.8387845943766823,
    "trainable_parameters": 1835008,
    "seconds": 27.078043792047538
  }
}
Published and verified 3fcd5e4922a749768d41fdb283d9b0f5
